# Exp 3 — Batched = unbatched, for every model, with no accuracy cost

Exp 2's fix made results **independent of batch-mates** (0.00% flips for all four detectors). But for RetinaNet and FCOS it moved *away* from the normal batch-1 output (−3.8 and −1.4 AP). The reason: those models were trained with torchvision's canvas-dependent anchor spacing, so "correcting" it broke what they had learned.

**The new fix, `self`,** takes a different target. Inside a batch, each image reproduces **its own normal batch-1 computation**:
- its valid area is its own batch-1 canvas (the image rounded up to a multiple of 32 for torchvision models; the image itself for DETR);
- its anchors are spaced the way its own batch-1 canvas would space them;
- GroupNorm statistics are taken over that same area;
- the padding beyond it is zeroed at every conv input, and scores there are removed;
- for Faster/Mask R-CNN, one border row and column is copied, so RoIAlign reads the image edge the way it does at batch 1.

If that works, batched output = normal unbatched output, for every image and every model.

**Speed.** Once batch-mates no longer matter, we can pick them for speed. Grouping similar shapes (sorted batches) pads less and runs faster. We compare against normal random batching.

**Models:** DETR, Faster R-CNN, Mask R-CNN (boxes), RetinaNet, FCOS — same 500 COCO images.

**Claims (fixed before running):**
- **F1 (exactness):** with `self`, batch-8 output (random, sorted, max-padding) differs from the normal batch-1 output on ≤ 0.5% of images, and AP is within 0.05, for every model.
- **F2 (invariance):** with `self`, batch-8 output differs from `self` batch-1 on ≤ 0.5% of images.
- **F3 (speed):** `self` with sorted batches is at least as fast as normal random batching (≥ 1.0×).

**Kaggle:** GPU T4 x2 or P100, Internet ON, nothing to attach. About 1–1.5 hours; resumes per model if restarted.

In [ ]:
!pip install -q "transformers==5.17.0" pycocotools timm

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation
import io, sys, glob, json, time, math, random, zipfile, platform, datetime, contextlib, urllib.request, dataclasses, gc
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torchvision
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
import transformers
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

OUT = "/kaggle/working/exp3"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

ENV = {
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if ENV["gpu"] is None:
    print("\n⚠⚠ NO GPU. Stop, set Accelerator to GPU in the right panel, and Run All again. ⚠⚠")

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

## 1. Configuration

In [ ]:
CFG = {
    "n_images": 500,
    "seed": 0,
    "batch_size": 8,
    "models": ["detr_r50", "frcnn_v2", "maskrcnn_v2", "retinanet_v2", "fcos"],
    "micro_batch": {"deformable_detr": 2},
    "flip_score_thr": 0.5,
    "flip_iou": 0.9,
    "coco_root": "/kaggle/input",
    "coco_download_dir": "/kaggle/working/coco",
    "claims": {"F1_flip": 0.005, "F1_dap": 0.05, "F2_flip": 0.005, "F3_speed": 1.0},
}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

## 2. COCO val2017 (same 500 images as before)

In [ ]:
def find_attached_coco(root):
    anns = glob.glob(os.path.join(root, "**", "instances_val2017.json"), recursive=True)
    probe = glob.glob(os.path.join(root, "**", "000000000139.jpg"), recursive=True)
    if anns and probe:
        return anns[0], os.path.dirname(probe[0])
    return None, None

ANN, IMG_DIR = find_attached_coco(CFG["coco_root"])
if ANN is None:
    os.makedirs(CFG["coco_download_dir"], exist_ok=True)
    ANN = os.path.join(CFG["coco_download_dir"], "instances_val2017.json")
    IMG_DIR = os.path.join(CFG["coco_download_dir"], "val2017")
    if not os.path.exists(ANN):
        zp = os.path.join(CFG["coco_download_dir"], "ann.zip")
        print("downloading COCO annotations (~250 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp) as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
        os.remove(zp)

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(ANN)
NAME2CAT = {c["name"].strip().lower(): c["id"] for c in GT.loadCats(GT.getCatIds())}
IDS = sorted(random.Random(CFG["seed"]).sample(sorted(GT.getImgIds()), CFG["n_images"]))
INFO = GT.loadImgs(IDS)
os.makedirs(IMG_DIR, exist_ok=True)
def fetch(info):
    p = os.path.join(IMG_DIR, info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{info['file_name']}", p)
    return p
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, INFO))
IMGS = [Image.open(p).convert("RGB") for p in paths]
N = len(IMGS)
print(f"{N} images; first ids {IDS[:3]} (Day 0 used seed 0 too, so these are the same images)")

## 3. The `self` fix

All switches off = the library's normal behaviour. `self` = the switches below, applied only when a batch actually contains padding beyond some image's own batch-1 canvas. A batch where every image already fills the canvas runs with no extra work.

In [ ]:
class Fix:
    COMPONENTS = ("conv", "norm", "own", "exactmask")
    def __init__(self):
        for c in self.COMPONENTS:
            setattr(self, c, False)
        self.valid, self.canvas, self.trivial, self._cache = None, None, True, {}

    def set(self, components):
        for c in self.COMPONENTS:
            setattr(self, c, c in components)

    def any_on(self):
        return any(getattr(self, c) for c in self.COMPONENTS)

    def begin(self, valid, canvas):
        self.valid = [tuple(int(v) for v in s) for s in valid]
        self.canvas = tuple(int(c) for c in canvas)
        self.trivial = all(v == self.canvas for v in self.valid)

    def end(self):
        self.valid = None

    def active(self):
        return self.valid is not None and not self.trivial

    def applies(self, x):
        return self.active() and x.dim() == 4 and x.shape[0] == len(self.valid)

    @staticmethod
    def _stride(n, N):
        for s in (1, 2, 4, 8, 16, 32, 64, 128, 256):
            if math.ceil(N / s) == n:
                return s
        return N / n

    def extents(self, H, W):
        """each image's valid (rows, cols) at a feature map of size H x W"""
        sh, sw = self._stride(H, self.canvas[0]), self._stride(W, self.canvas[1])
        return [(math.ceil(h / sh), math.ceil(w / sw)) for h, w in self.valid]

    def mask(self, x):
        H, W = x.shape[-2:]
        key = (H, W, tuple(self.valid), self.canvas, x.device, x.dtype)
        if key not in self._cache:
            m = torch.zeros(len(self.valid), 1, H, W, device=x.device, dtype=x.dtype)
            for i, (h, w) in enumerate(self.extents(H, W)):
                m[i, :, :h, :w] = 1
            if len(self._cache) > 64:
                self._cache.clear()
            self._cache[key] = m
        return self._cache[key]

def _mask_conv_inputs(root, fix):
    """Zero the padding at the input of every conv / max-pool: each layer then sees exactly the zeros its
    batch-1 run sees past the canvas edge (activations are >= 0 after ReLU, so max-pool is unaffected)."""
    def pre(mod, inp):
        x = inp[0]
        if fix.conv and fix.applies(x):
            return (x * fix.mask(x),) + tuple(inp[1:])
    for m in root.modules():
        if isinstance(m, (nn.Conv2d, nn.MaxPool2d)):
            m.register_forward_pre_hook(pre)

def _masked_groupnorm(root, fix):
    """GroupNorm statistics over each image's own valid area only."""
    n = 0
    for mod in root.modules():
        if isinstance(mod, nn.GroupNorm):
            orig = mod.forward
            def fwd(x, mod=mod, orig=orig):
                if not (fix.norm and fix.applies(x)):
                    return orig(x)
                m = fix.mask(x); B, C, H, W = x.shape; G = mod.num_groups
                xg, mg = x.reshape(B, G, C // G, H, W), m.reshape(B, 1, 1, H, W)
                cnt = mg.sum(dim=(2, 3, 4), keepdim=True) * (C // G)
                mean = (xg * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                var = (((xg - mean) ** 2) * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                y = ((xg - mean) / torch.sqrt(var + mod.eps)).reshape(B, C, H, W)
                if mod.affine:
                    y = y * mod.weight[None, :, None, None] + mod.bias[None, :, None, None]
                return y
            mod.forward = fwd; n += 1
    return n

def _suppress_padding_scores(conv, fix):
    """Classification logits at positions that do not exist in the image's batch-1 run -> -1e4."""
    def hook(mod, inp, out):
        if fix.conv and fix.applies(out):
            return out.masked_fill(fix.mask(out) == 0, -1e4)
    conv.register_forward_hook(hook)

def _own_anchors(ag, fix):
    """Each image gets the anchors its own batch-1 canvas would give: torchvision spaces anchors at
    canvas // grid, so we use the image's own canvas and grid. Anchors at positions that do not exist in
    its batch-1 run become zero-size boxes, which torchvision removes."""
    orig = ag.forward
    def fwd(image_list, feature_maps):
        if not (fix.own and fix.active()):
            return orig(image_list, feature_maps)
        grid_sizes = [list(f.shape[-2:]) for f in feature_maps]
        Hc, Wc = image_list.tensors.shape[-2:]
        dtype, device = feature_maps[0].dtype, feature_maps[0].device
        ag.set_cell_anchors(dtype, device)
        n_cell = [ca.shape[0] for ca in ag.cell_anchors]
        out = []
        for ch, cw in fix.valid:
            strides, keep = [], []
            for (gh, gw), a in zip(grid_sizes, n_cell):
                sh, sw = 2 ** round(math.log2(Hc / gh)), 2 ** round(math.log2(Wc / gw))
                oh, ow = math.ceil(ch / sh), math.ceil(cw / sw)
                strides.append([torch.tensor(ch // oh, dtype=torch.int64, device=device),
                                torch.tensor(cw // ow, dtype=torch.int64, device=device)])
                k = torch.zeros(gh, gw, dtype=dtype, device=device); k[:oh, :ow] = 1
                keep.append(k.reshape(-1).repeat_interleave(a))
            anc = ag.grid_anchors(grid_sizes, strides)
            out.append(torch.cat([x * k[:, None] for x, k in zip(anc, keep)]))
        return out
    ag.forward = fwd

def install_fix_tv(model, fix):
    heads = [model.rpn.head] if hasattr(model, "rpn") else [model.head]
    for root in [model.backbone] + heads:
        _mask_conv_inputs(root, fix)
    has_roi = hasattr(model, "roi_heads")
    def zero_outputs(mod, inp, out):
        if not (fix.conv and fix.active()):
            return
        new = type(out)()
        for k, v in out.items():
            v = v * fix.mask(v)
            if fix.own and has_roi:
                # RoIAlign at batch 1 clamps samples at the canvas edge to the last row/column; copying that
                # row/column one step outward reproduces this inside a bigger canvas.
                H, W = v.shape[-2:]
                for i, (hl, wl) in enumerate(fix.extents(H, W)):
                    if wl < W:
                        v[i, :, :hl, wl] = v[i, :, :hl, wl - 1]
                    if hl < H:
                        v[i, :, hl, :min(wl + 1, W)] = v[i, :, hl - 1, :min(wl + 1, W)]
            new[k] = v
        return new
    model.backbone.register_forward_hook(zero_outputs)
    g = sum(_masked_groupnorm(h, fix) for h in heads) + _masked_groupnorm(model.backbone, fix)
    if hasattr(model, "rpn"):
        _own_anchors(model.rpn.anchor_generator, fix); _suppress_padding_scores(model.rpn.head.cls_logits, fix)
    else:
        _own_anchors(model.anchor_generator, fix); _suppress_padding_scores(model.head.classification_head.cls_logits, fix)
    fix.SELF = ("conv", "norm", "own") if g else ("conv", "own")

def install_fix_hf(model, fix):
    enc = model.model.backbone
    _mask_conv_inputs(enc, fix)
    orig = enc.forward
    def fwd(pixel_values, pixel_mask):
        out = orig(pixel_values, pixel_mask)
        if not (fix.exactmask and fix.active()):
            return out
        # the transformer's padding mask with exact extents (the library shrinks it with nearest-neighbour rounding)
        return [(fm, fix.mask(fm)[:, 0].bool()) for fm, _ in out]
    enc.forward = fwd
    fix.SELF = ("conv", "exactmask")

## 4. Detectors

In [ ]:
ALIASES = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch", "tvmonitor": "tv", "tv monitor": "tv",
           "diningtable": "dining table", "pottedplant": "potted plant", "hair dryer": "hair drier",
           "sports ball": "sports ball", "cellphone": "cell phone", "mobile phone": "cell phone"}
def _norm(s):
    s = " ".join(str(s).lower().replace("_", " ").replace("-", " ").split())
    return ALIASES.get(s, s)
def names_to_cat(names, label=""):
    m, miss = {}, []
    for k, v in names.items():
        c = NAME2CAT.get(_norm(v))
        if c is not None:
            m[int(k)] = c
        elif _norm(v) not in ("n/a", "__background__", "background", "no object"):
            miss.append(v)
    if miss:
        print(f"  unmapped class names in {label}: {miss}")
    return m

def load_hf_model(cls, repo, revision=None):
    try:
        return cls.from_pretrained(repo, revision=revision)
    except Exception as ex:
        if "alidation" not in f"{type(ex).__name__} {ex}":
            raise
        from huggingface_hub import hf_hub_download
        raw = json.load(open(hf_hub_download(repo, "config.json", revision=revision)))
        defaults = {f.name: f.default for f in dataclasses.fields(cls.config_class) if f.default is not dataclasses.MISSING}
        dropped = [k for k, v in raw.items() if v is None and defaults.get(k) is not None]
        print(f"  (config fix for {repo}: replaced null fields {dropped} with library defaults)")
        raw = {k: v for k, v in raw.items() if k not in dropped}
        return cls.from_pretrained(repo, revision=revision, config=cls.config_class.from_dict(raw))

def roundup(x, d):
    return int(math.ceil(x / d) * d)

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

def _pack(boxes, scores, labels, lab2cat):
    lab = labels.cpu().numpy()
    keep = np.array([l in lab2cat for l in lab], bool)
    return {"boxes": boxes.float().cpu().numpy()[keep], "scores": scores.float().cpu().numpy()[keep],
            "labels": np.array([lab2cat[l] for l in lab[keep]], dtype=int)}

class MicroBatch:
    """If a batch does not fit in GPU memory, split it into smaller pieces that are all padded to the canvas the
    full batch would have had. Images in these models never interact except through padding, so this gives the
    same result as the full batch (verified for DETR by the canvas self-test, and for every Part-B model by B0)."""
    micro = None   # largest piece size known to fit; None = no limit found yet

    def batch_canvas(self, imgs):
        sizes = [self.input_size(im) for im in imgs]
        return (roundup(max(h for h, _ in sizes), self.div), roundup(max(w for _, w in sizes), self.div))

    def run(self, imgs, canvas=None):
        if self.fixed_size:
            return self._run(imgs, canvas)
        if self.micro is not None and len(imgs) > self.micro:
            return self._split(imgs, canvas)
        try:
            return self._run(imgs, canvas)
        except torch.cuda.OutOfMemoryError:
            if len(imgs) == 1:
                raise
        # We are now OUTSIDE the except block, so the failed attempt (and the GPU memory it holds) is released
        # before retrying. Retrying inside the except block keeps that memory alive.
        free_mem()
        self.micro = max(1, len(imgs) // 2)
        print(f"  [{self.name}] batch of {len(imgs)} does not fit in GPU memory → running pieces of {self.micro} "
              f"on the full batch's canvas (equivalent result)")
        return self._split(imgs, canvas)

    def _split(self, imgs, canvas):
        canvas = canvas if canvas is not None else self.batch_canvas(imgs)
        step = self.micro                      # fixed for this call, even if a piece lowers self.micro further
        R, P = [], []
        for k in range(0, len(imgs), step):
            r, p = self.run(imgs[k:k + step], canvas)
            R += r; P += p
        assert len(R) == len(imgs)
        return R, P

class HFDet(MicroBatch):
    kind, div = "hf", 1
    def __init__(self, name, repo, cls_name, revision=None, fixed_size=False):
        from transformers import AutoImageProcessor
        self.name, self.fixed_size = name, fixed_size
        self.proc = AutoImageProcessor.from_pretrained(repo, revision=revision)
        self.model = load_hf_model(getattr(transformers, cls_name), repo, revision).to(DEVICE).eval()
        self.lab2cat = names_to_cat(self.model.config.id2label, name)
        self.fix = Fix()
        if not fixed_size:
            install_fix_hf(self.model, self.fix)

    def _single(self, img):
        enc = self.proc(images=[img], return_tensors="pt")
        pv = enc["pixel_values"][0]
        m = enc["pixel_mask"][0] if "pixel_mask" in enc else torch.ones(pv.shape[-2:], dtype=torch.long)
        return pv, m

    def input_size(self, img):
        return tuple(self._single(img)[0].shape[-2:])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        if canvas is None:
            enc = self.proc(images=imgs, return_tensors="pt")
            pv = enc["pixel_values"]
            mask = enc["pixel_mask"] if "pixel_mask" in enc else None
        else:
            H, W = canvas; pvs, ms = [], []
            for im in imgs:
                p, m = self._single(im); h, w = p.shape[-2:]
                assert h <= H and w <= W, f"canvas {canvas} smaller than image {(h, w)}"
                pvs.append(F.pad(p, (0, W - w, 0, H - h))); ms.append(F.pad(m, (0, W - w, 0, H - h)))
            pv, mask = torch.stack(pvs), torch.stack(ms)
        kw = {"pixel_values": pv.to(DEVICE)}
        if mask is not None:
            kw["pixel_mask"] = mask.to(DEVICE)
            if self.fix.any_on():
                self.fix.begin([(int(mask[i, :, 0].sum()), int(mask[i, 0, :].sum())) for i in range(len(imgs))], pv.shape[-2:])
        try:
            out = self.model(**kw)
        finally:
            self.fix.end()
        sizes = torch.tensor([[im.size[1], im.size[0]] for im in imgs])
        res = self.proc.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        pad = (1 - mask.float().mean(dim=(1, 2))).tolist() if mask is not None else [0.0] * len(imgs)
        return [_pack(r["boxes"], r["scores"], r["labels"], self.lab2cat) for r in res], pad

class TVDet(MicroBatch):
    kind, div, fixed_size = "tv", 32, False
    def __init__(self, name, builder, weights):
        self.name = name
        self.model = builder(weights=weights).to(DEVICE).eval()
        self.tf = weights.transforms()
        self.lab2cat = names_to_cat(dict(enumerate(weights.meta["categories"])), name)
        self._setup()

    def _setup(self):
        self._canvas = None
        self.fix = Fix()
        orig = self.model.transform.batch_images
        def batch_images(images, size_divisible=32):
            b = orig(images, size_divisible)
            H, W = self._canvas if self._canvas is not None else tuple(b.shape[-2:])
            assert b.shape[-2] <= H and b.shape[-1] <= W, f"canvas {(H, W)} smaller than batch {tuple(b.shape[-2:])}"
            if (H, W) == tuple(b.shape[-2:]):
                return b
            return F.pad(b, (0, W - b.shape[-1], 0, H - b.shape[-2]))
        self.model.transform.batch_images = batch_images
        install_fix_tv(self.model, self.fix)

    @torch.inference_mode()
    def input_size(self, img):
        il, _ = self.model.transform([self.tf(img).to(DEVICE)])
        return tuple(il.image_sizes[0])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        self._canvas = canvas
        try:
            x = [self.tf(im).to(DEVICE) for im in imgs]
            il, _ = self.model.transform(x)
            H, W = il.tensors.shape[-2:]
            pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
            if self.fix.any_on():
                d = self.model.transform.size_divisible
                sizes = [tuple(s) for s in il.image_sizes]
                self.fix.begin([(roundup(h, d), roundup(w, d)) for h, w in sizes] if self.fix.own else sizes, (H, W))
            out = self.model(x)
        finally:
            self._canvas = None; self.fix.end()
        return [_pack(o["boxes"], o["scores"], o["labels"], self.lab2cat) for o in out], pad

def _tv(builder_name, weights_name, name):
    import torchvision.models.detection as D
    return lambda: TVDet(name, getattr(D, builder_name), getattr(D, weights_name).DEFAULT)

LOADERS = {
    "detr_r50": lambda: HFDet("detr_r50", "facebook/detr-resnet-50", "DetrForObjectDetection", revision="no_timm"),
    "deformable_detr": lambda: HFDet("deformable_detr", "SenseTime/deformable-detr", "DeformableDetrForObjectDetection"),
    "frcnn_v2": _tv("fasterrcnn_resnet50_fpn_v2", "FasterRCNN_ResNet50_FPN_V2_Weights", "frcnn_v2"),
    "maskrcnn_v2": _tv("maskrcnn_resnet50_fpn_v2", "MaskRCNN_ResNet50_FPN_V2_Weights", "maskrcnn_v2"),
    "retinanet_v2": _tv("retinanet_resnet50_fpn_v2", "RetinaNet_ResNet50_FPN_V2_Weights", "retinanet_v2"),
    "fcos": _tv("fcos_resnet50_fpn", "FCOS_ResNet50_FPN_Weights", "fcos"),
    "rtdetr_r50": lambda: HFDet("rtdetr_r50", "PekingU/rtdetr_r50vd", "RTDetrForObjectDetection", fixed_size=True),
}
# Published full COCO val2017 box AP; our 500-image batch-1 AP should be within about ±2.
REF_AP = {"detr_r50": 42.0, "deformable_detr": 44.5, "frcnn_v2": 46.7, "maskrcnn_v2": 47.4,
          "retinanet_v2": 41.5, "fcos": 39.2, "rtdetr_r50": 53.1}

def load(name):
    free_mem()
    t0 = time.time()
    try:
        det = LOADERS[name]()
        det.micro = CFG["micro_batch"].get(name)
    except Exception as ex:
        print(f"✗ {name} failed to load: {type(ex).__name__}: {str(ex)[:300]}  — skipping this model")
        return None
    note = f"; runs in pieces of {det.micro} (padded to the full batch's canvas)" if det.micro else ""
    print(f"\n== {name} loaded in {time.time() - t0:.0f}s; mapped classes {len(set(det.lab2cat.values()))}/80{note}")
    return det

def free_mem():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 5. Metrics and helpers

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(p, q, thr, iou_min=0.5):
    ia = np.where(p["scores"] >= thr)[0]; ib = np.where(q["scores"] >= thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bc = -1.0, None
        for c, j in enumerate(ib):
            if c not in used and p["labels"][i] == q["labels"][j] and M[r, c] > best:
                best, bc = M[r, c], c
        if bc is not None and best >= iou_min:
            used.add(bc); pairs.append((i, ib[bc], best))
    return pairs, ia, ib

def compare(ref, new):
    """Image-level flip + detection-level change counts + max score change, ref vs new."""
    pairs, ia, ib = greedy_match(ref, new, CFG["flip_score_thr"])
    stable = sum(iou >= CFG["flip_iou"] for _, _, iou in pairs)
    moved = len(pairs) - stable
    lost, gained = len(ia) - len(pairs), len(ib) - len(pairs)
    allp, _, _ = greedy_match(ref, new, 0.05)
    dscore = max([abs(ref["scores"][i] - new["scores"][j]) for i, j, _ in allp], default=0.0)
    return {"flip": (moved + lost + gained) > 0, "stable": stable, "moved": moved, "lost": lost, "gained": gained,
            "max_score_delta": dscore}

def det_survival(ref, new, W, H):
    """For each confident ref detection: did it survive unchanged? plus its distance to the right/bottom edge."""
    pairs, ia, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ok = {i for i, _, iou in pairs if iou >= CFG["flip_iou"]}
    rows = []
    for i in ia:
        x1, y1, x2, y2 = ref["boxes"][i]
        rows.append({"changed": i not in ok, "d_right": (W - x2) / W, "d_bottom": (H - y2) / H,
                     "rel_area": (x2 - x1) * (y2 - y1) / (W * H), "score": ref["scores"][i]})
    return rows

def coco_ap(preds, idx=None):
    idx = list(range(N)) if idx is None else idx
    dets = []
    for k in idx:
        r = preds[k]; order = np.argsort(-r["scores"])[:100]
        for t in order:
            x1, y1, x2, y2 = r["boxes"][t]
            dets.append({"image_id": IDS[k], "category_id": int(r["labels"][t]),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(r["scores"][t])})
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "bbox"); E.params.imgIds = [IDS[k] for k in idx]
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

def run_batches(det, batches, canvas_fn=None, targets=None):
    """batches: list of index lists. canvas_fn(batch)->(H,W) or None. targets: optional list of index lists
    saying which members of each batch to record (others are companions). Returns preds, pads, seconds."""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([IMGS[i] for i in b], canvas=canvas_fn(b) if canvas_fn else None)
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

In [ ]:
B = CFG["batch_size"]
def chunks(order, k=B):
    order = list(order)
    return [order[i:i + k] for i in range(0, len(order), k)]

## 6. Run: normal vs `self`, at batch 1 and in three kinds of batch-8

In [ ]:
def warmup(det):
    for _ in range(2):
        det.run(IMGS[:B]); det.run(IMGS[:1])
    sync()

bs1 = [[i] for i in range(N)]
rand0 = chunks(np.random.default_rng(0).permutation(N))
aspect = np.array([im.size[0] / im.size[1] for im in IMGS]); area = np.array([im.size[0] * im.size[1] for im in IMGS])
sorted_b = chunks(np.lexsort((area, aspect)))
WIDEST, TALLEST = int(np.argmax(aspect)), int(np.argmin(aspect))
mp = chunks(range(N), B - 2); maxpad = [c + [WIDEST, TALLEST] for c in mp]
PLAN = [("none", "bs1", bs1, None), ("none", "bs8_rand", rand0, None), ("none", "bs8_sorted", sorted_b, None),
        ("self", "bs1", bs1, None), ("self", "bs8_rand", rand0, None), ("self", "bs8_sorted", sorted_b, None),
        ("self", "bs8_maxpad", maxpad, mp)]

R = read_csv(f"{OUT}/runs.csv").to_dict("records") if os.path.exists(f"{OUT}/runs.csv") else []
PI = read_csv(f"{OUT}/per_image.csv").to_dict("records") if os.path.exists(f"{OUT}/per_image.csv") else []
done = {r["model"] for r in R}
for mname in CFG["models"]:
    if mname in done:
        print(f"== {mname}: already done, skipping"); continue
    det = None; free_mem()
    det = load(mname)
    if det is None:
        continue
    warmup(det)
    preds = {}
    for vname, bname, batches, targets in PLAN:
        det.fix.set(det.fix.SELF if vname == "self" else ())
        P, PD, sec = run_batches(det, batches, targets=targets)
        preds[(vname, bname)] = P
        ap = coco_ap(P)
        R.append({"model": mname, "variant": vname, "batching": bname, "AP": ap, "img_per_s": N / sec,
                  "mean_pad": float(np.mean(PD))})
        print(f"  {vname:5s} {bname:11s} AP={ap:6.2f}  pad={np.mean(PD):.3f}  {N / sec:5.1f} img/s")
    det.fix.set(())
    ref = preds[("none", "bs1")]
    for (vname, bname), P in preds.items():
        if (vname, bname) == ("none", "bs1"):
            continue
        for i in range(N):
            PI.append({"model": mname, "variant": vname, "batching": bname, "pair": "vs_default_bs1", **compare(ref[i], P[i])})
            if vname == "self" and bname != "bs1":
                PI.append({"model": mname, "variant": vname, "batching": bname, "pair": "vs_self_bs1",
                           **compare(preds[("self", "bs1")][i], P[i])})
    det = None; preds = None; free_mem()
    pd.DataFrame(R).to_csv(f"{OUT}/runs.csv", index=False)
    pd.DataFrame(PI).to_csv(f"{OUT}/per_image.csv", index=False)

## 7. Results

In [ ]:
R = read_csv(f"{OUT}/runs.csv"); PI = read_csv(f"{OUT}/per_image.csv")
fl = PI.groupby(["model", "variant", "batching", "pair"]).agg(flip=("flip", "mean"),
        max_dscore=("max_score_delta", "max")).reset_index()
base_ap = R[(R.variant == "none") & (R.batching == "bs1")].set_index("model")["AP"]
base_rand = R[(R.variant == "none") & (R.batching == "bs8_rand")].set_index("model")["img_per_s"]
base_bs1 = R[(R.variant == "none") & (R.batching == "bs1")].set_index("model")["img_per_s"]
R["dAP_vs_default_bs1"] = R.apply(lambda r: r.AP - base_ap[r.model], axis=1)
R["speed_vs_default_bs8_rand"] = R.apply(lambda r: r.img_per_s / base_rand[r.model], axis=1)
R["speed_vs_default_bs1"] = R.apply(lambda r: r.img_per_s / base_bs1[r.model], axis=1)
T = R.merge(fl[fl.pair == "vs_default_bs1"][["model", "variant", "batching", "flip", "max_dscore"]],
            on=["model", "variant", "batching"], how="left")
display(T[["model", "variant", "batching", "AP", "dAP_vs_default_bs1", "flip", "max_dscore", "img_per_s",
           "speed_vs_default_bs8_rand", "speed_vs_default_bs1"]].style.format(
    {"AP": "{:.2f}", "dAP_vs_default_bs1": "{:+.2f}", "flip": "{:.2%}", "max_dscore": "{:.1e}", "img_per_s": "{:.1f}",
     "speed_vs_default_bs8_rand": "{:.2f}×", "speed_vs_default_bs1": "{:.2f}×"}, na_rep="—"))
print("'flip' and 'max_dscore' compare every run with the library's normal batch-1 output.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
models = list(dict.fromkeys(R.model))
cols = [("none", "bs8_rand"), ("none", "bs8_sorted"), ("self", "bs8_rand"), ("self", "bs8_sorted"), ("self", "bs8_maxpad")]
w = 0.16
for k, (v, b) in enumerate(cols):
    d = T[(T.variant == v) & (T.batching == b)].set_index("model").reindex(models)
    axes[0].bar(np.arange(len(models)) + (k - 2) * w, d["flip"] * 100, w, label=f"{v} {b}")
    axes[1].bar(np.arange(len(models)) + (k - 2) * w, d["speed_vs_default_bs8_rand"], w, label=f"{v} {b}")
for ax, yl in [(axes[0], "% images differing from normal batch-1 output"), (axes[1], "throughput vs normal random batch-8")]:
    ax.set_xticks(range(len(models))); ax.set_xticklabels(models, rotation=15); ax.set_ylabel(yl); ax.legend(fontsize=8)
axes[1].axhline(1.0, c="k", lw=1, ls="--")
axes[0].set_title("Exactness: does batching change the answer?"); axes[1].set_title("Speed")
plt.tight_layout(); plt.savefig(f"{OUT}/exp3_summary.png", dpi=180); plt.show()

## 8. Claims check

In [ ]:
K = CFG["claims"]; v = {}
f1, f2, f3 = [], [], []
for m in R.model.unique():
    s = T[(T.model == m) & (T.variant == "self") & (T.batching != "bs1")]
    ok1 = (s.flip.max() <= K["F1_flip"]) and (s.dAP_vs_default_bs1.abs().max() <= K["F1_dap"])
    inv = fl[(fl.model == m) & (fl.variant == "self") & (fl.pair == "vs_self_bs1")].flip.max()
    spd = R[(R.model == m) & (R.variant == "self") & (R.batching == "bs8_sorted")].speed_vs_default_bs8_rand.iloc[0]
    f1.append(ok1); f2.append(inv <= K["F2_flip"]); f3.append(spd >= K["F3_speed"])
    print(f"  {m:13s} self vs normal bs1: worst flip {s.flip.max():.2%}, worst ΔAP {s.dAP_vs_default_bs1.abs().max():.3f} | "
          f"self bs8 vs self bs1: {inv:.2%} | sorted speed {spd:.2f}× of normal random bs8")
v["F1_exactness"] = f"{'✓' if all(f1) else '✗'}  {sum(f1)}/{len(f1)} models"
v["F2_invariance"] = f"{'✓' if all(f2) else '✗'}  {sum(f2)}/{len(f2)} models"
v["F3_speed"] = f"{'✓' if all(f3) else '✗'}  {sum(f3)}/{len(f3)} models"
for k, x in v.items():
    print(f"{k:14s} {x}")
with open(f"{OUT}/manifest.json", "w") as f:
    json.dump({"env": ENV, "config": CFG, "image_ids": IDS, "verdicts": v}, f, indent=2, default=str)
print("saved:", sorted(os.listdir(OUT)))

## What to bring back

Upload the finished notebook (or the `exp3` folder). Key outputs: the results table, the summary figure, and the claim lines.